# `apptbl-covariates`: respondent covariates in the decision-model input, by dataset

In [1]:
from pathlib import Path
import sys

from IPython.display import display
import pandas as pd

ASSET_NAME = "apptbl-covariates"
ASSET_DIR = Path.cwd()
PAPER_DIR = ASSET_DIR.parent
if not (PAPER_DIR / "microdata_analysis.py").is_file():
    raise FileNotFoundError(f"Open this notebook from manuscript-training-assets/; {PAPER_DIR} has no microdata_analysis.py")
if str(PAPER_DIR) not in sys.path:
    sys.path.insert(0, str(PAPER_DIR))

import microdata_analysis as ma

print(f"Outputs go to {ASSET_DIR / ASSET_NAME}.*")

Outputs go to /home/dw/github/decision-models-finetuning/code/train-notebooks/beta-training/ms_paper_beta_loss/manuscript-training-assets/apptbl-covariates.*


## Datasets

In [2]:
ma.setup_analysis_env()
from beta_ols import berke2023 as ds_berke
from beta_ols import covid as ds_covid
from beta_ols import grummon2025 as ds_grummon
from beta_ols import kantorowicz2026 as ds_kantorowicz
from beta_ols import kriner2026 as ds_kriner
from beta_ols import magistro2025 as ds_magistro
from beta_ols import wu2025 as ds_wu

# Column order of the table. Training and evaluation build each respondent's covariate lines from
# module.PERSONA_COLS with include_country=False (beta_train/items.py, beta_train/validation.py).
DATASET_MODULES = {
    "Electricity": ds_wu,
    "Drones": ds_berke,
    "Flights": ds_kriner,
    "Consumer rights": ds_kantorowicz,
    "AI & offshoring": ds_magistro,
    "Food labels": ds_grummon,
    "Vaccines": ds_covid,
}
COUNTRY_COLUMN = "c_alphan"

## Labels and formatting (edit wording here)

In [3]:
# Row order and the short name shown for each covariate column.
COVARIATE_LABELS = {
    "SEX": "Sex",
    "AGE": "Age",
    "EDULEVEL": "Education",
    "US_INC": "Income",
    "RACE": "Race",
    "HISPANIC": "Hispanic or Latino",
    "US_REG": "Region",
    "URBRURAL": "Place of living",
    "RELIGGRP": "Religion",
    "US_PRTY": "Political party",
    "PARTY_LR": "Political ideology",
    "POL_LABEL": "Political self-label",
    "VOTE_LE": "Voted in last election",
    "MARITAL": "Marital status",
    "MAINSTAT": "Main activity",
    "ORIENTATION": "Sexual orientation",
    "MILITARY": "Military service",
    "RESTYPE": "Residence type",
    "OWNHOME": "Owns residence",
    "RENTHOME": "Rents residence",
    "ELECHEAT": "Electric heating",
    "HHSIZE": "Household size",
    "CHILDREN": "Children in household",
    "SNAP": "SNAP recipient",
    "WIC": "WIC recipient",
    "NFPUSE": "Nutrition Facts label use",
    "ONLINEAMT": "Online grocery share",
    "HEALTHLIT": "Nutrition literacy score",
}
STUB_HEADER = "Covariate"
MARK = "✓"
TOTAL_LABEL = "**Total**"

## Format as a Pandoc pipe table

In [4]:
import math
import re

from matplotlib import font_manager
from PIL import ImageFont

# Page and table style of manuscript/quarto-files/ms_template.docx. Word fixes the column widths that Pandoc
# derives from the rule dash counts, so a column narrower than its longest word splits that word across lines.
TEXT_WIDTH_PT = 468.0  # 8.5 in page minus 1 in left and right margins
TABLE_FONT_PT = 10.0  # Compact paragraph style used in table cells
CELL_PADDING_PT = 10.8  # Table style left + right cell margins (108 twips each)
COLUMN_SLACK_PT = 2.0  # per-column allowance for differences between Word and the font metrics measured here
MATH_WIDTH_FACTOR = 1.15  # Word sets $...$ in Cambria Math, which runs wider than Times New Roman
# No-break spaces around ×: Word sets each $...$ as one equation that never wraps.
MATH_SYMBOLS = {r"\beta": "β", r"\rho": "ρ", r"\times": "\u00a0×\u00a0"}


def load_table_font(weight):
    font_properties = font_manager.FontProperties(family="Times New Roman", weight=weight)
    font_path = font_manager.findfont(font_properties, fallback_to_default=False)
    return ImageFont.truetype(font_path, 1000)


REGULAR_FONT = load_table_font("normal")
BOLD_FONT = load_table_font("bold")


def shown_math(match):
    latex = match.group(1)
    text = latex
    for command, symbol in MATH_SYMBOLS.items():
        text = text.replace(command, symbol)
    text = text.replace(r"\mathrm", "")
    for mark in ("{", "}", "_", "^"):
        text = text.replace(mark, "")
    if "\\" in text:
        raise ValueError(f"No display text for a LaTeX command in {latex!r}; add it to MATH_SYMBOLS")
    return text


def shown_citation(match):
    surname = match.group(1).title()
    year = match.group(2)
    return f"({surname} {year})"


def shown_cell(cell):
    """Text Word displays for a markdown cell, with the font and width factor to measure it."""
    is_bold = cell.startswith("**") and cell.endswith("**")
    text = cell
    if is_bold:
        text = text[2:-2]
    text = re.sub(r"\[@([a-z]+)(\d{4})[a-z0-9]*\]", shown_citation, text)
    has_math = re.search(r"(?<!\\)\$", text) is not None
    text = re.sub(r"(?<!\\)\$(.+?)(?<!\\)\$", shown_math, text)
    text = text.replace(r"\$", "$")
    text = text.replace(r"\|", "|")
    font = BOLD_FONT if is_bold else REGULAR_FONT
    factor = MATH_WIDTH_FACTOR if has_math else 1.0
    return text, font, factor


def text_width_pt(text, font, factor):
    return font.getlength(text) / 1000.0 * TABLE_FONT_PT * factor


def unbreakable_pieces(text):
    """Pieces Word keeps on one line: words, with compound words (letter-hyphen-letter) split after the hyphen."""
    pieces = []
    for word in re.split(r"[ \t\n]+", text):
        pieces.extend(re.split(r"(?<=[A-Za-z]-)(?=[A-Za-z])", word))
    return pieces


def column_widths_pt(headers, rows):
    """Each column gets at least its widest unbreakable piece; leftover page width goes to columns that wrap."""
    n_columns = len(headers)
    min_pt = [0.0] * n_columns
    natural_pt = [0.0] * n_columns
    for cells in [headers, *rows]:
        if len(cells) != n_columns:
            raise ValueError(f"Row has {len(cells)} cells, expected {n_columns}: {cells}")
        for column_index, cell in enumerate(cells):
            text, font, factor = shown_cell(cell)
            for piece in unbreakable_pieces(text):
                piece_pt = text_width_pt(piece, font, factor)
                min_pt[column_index] = max(min_pt[column_index], piece_pt)
            cell_pt = text_width_pt(text, font, factor)
            natural_pt[column_index] = max(natural_pt[column_index], cell_pt)
    extra_pt = CELL_PADDING_PT + COLUMN_SLACK_PT
    min_pt = [width + extra_pt for width in min_pt]
    natural_pt = [width + extra_pt for width in natural_pt]
    if sum(min_pt) > TEXT_WIDTH_PT:
        raise ValueError(
            f"Longest words need {sum(min_pt):.0f}pt but the page is {TEXT_WIDTH_PT:.0f}pt wide; "
            f"shorten labels or drop columns. Per-column minimum: {[round(width) for width in min_pt]}"
        )
    if sum(natural_pt) <= TEXT_WIDTH_PT:
        return natural_pt
    spare_pt = TEXT_WIDTH_PT - sum(min_pt)
    stretch_pt = [natural - minimum for natural, minimum in zip(natural_pt, min_pt)]
    total_stretch_pt = sum(stretch_pt)
    widths_pt = []
    for minimum, stretch in zip(min_pt, stretch_pt):
        widths_pt.append(minimum + spare_pt * stretch / total_stretch_pt)
    return widths_pt


def escape_markdown(text):
    """Escape $ and | so Pandoc reads them as text inside a pipe table."""
    escaped = text.replace("$", r"\$")
    escaped = escaped.replace("|", r"\|")
    return escaped


def pipe_table(headers, rows, column_rules):
    """Pandoc pipe table. column_rules set alignment (':---' left, '---:' right); dash counts come from column_widths_pt."""
    if len(column_rules) != len(headers):
        raise ValueError(f"{len(column_rules)} column rules for {len(headers)} columns")
    widths_pt = column_widths_pt(headers, rows)
    rule_cells = []
    for column_rule, width_pt in zip(column_rules, widths_pt):
        if re.fullmatch(r":?-+:?", column_rule) is None:
            raise ValueError(f"Bad column rule {column_rule!r}")
        dashes = "-" * round(width_pt)
        left = ":" if column_rule.startswith(":") else ""
        right = ":" if column_rule.endswith(":") else ""
        rule_cells.append(left + dashes + right)
    lines = ["| " + " | ".join(headers) + " |"]
    lines.append("|" + "|".join(rule_cells) + "|")
    for row in rows:
        lines.append("| " + " | ".join(row) + " |")
    return "\n".join(lines) + "\n"

## Build the table

In [5]:
covariates_by_dataset = {}
for dataset_label, module in DATASET_MODULES.items():
    choice_df = module.load_choice_df()
    covariates = [column for column in module.PERSONA_COLS if column != COUNTRY_COLUMN]
    for column in covariates:
        if column not in choice_df.columns:
            raise KeyError(f"{dataset_label}: covariate {column} is not a column of the choice data")
        if choice_df[column].isna().all():
            raise ValueError(f"{dataset_label}: covariate {column} is empty for every respondent")
        if column not in COVARIATE_LABELS:
            raise KeyError(f"{dataset_label}: no label for covariate {column}; add it to COVARIATE_LABELS")
    covariates_by_dataset[dataset_label] = set(covariates)

used_covariates = set()
for covariates in covariates_by_dataset.values():
    used_covariates |= covariates
unused_labels = [column for column in COVARIATE_LABELS if column not in used_covariates]
if unused_labels:
    raise ValueError(f"COVARIATE_LABELS has covariates no dataset uses: {unused_labels}")

headers = [STUB_HEADER, *DATASET_MODULES]
rows = []
for column, covariate_label in COVARIATE_LABELS.items():
    row = [covariate_label]
    for dataset_label in DATASET_MODULES:
        if column in covariates_by_dataset[dataset_label]:
            row.append(MARK)
        else:
            row.append("")
    rows.append(row)
total_row = [TOTAL_LABEL]
for dataset_label in DATASET_MODULES:
    total_row.append(f"**{len(covariates_by_dataset[dataset_label])}**")
rows.append(total_row)

column_rules = [":---", *[":---:" for _ in DATASET_MODULES]]
table_md = pipe_table(headers, rows, column_rules)
display(pd.DataFrame(rows, columns=headers))

,Covariate,Electricity,Drones,Flights,Consumer rights,AI & offshoring,Food labels,Vaccines
0,Sex,✓,✓,✓,✓,✓,✓,✓
1,Age,✓,✓,✓,✓,✓,✓,✓
2,Education,✓,,✓,✓,✓,✓,✓
3,Income,✓,✓,✓,✓,✓,✓,✓
4,Race,✓,✓,✓,,,✓,✓
5,Hispanic or Latino,✓,,,,,✓,
6,Region,✓,✓,✓,,✓,,✓
7,Place of living,,✓,✓,,,,
8,Religion,,,✓,,,,✓
9,Political party,✓,,✓,,✓,,✓


## Save `apptbl-covariates.md` (included by the manuscript)

In [6]:
table_path = ASSET_DIR / f"{ASSET_NAME}.md"
table_path.write_text(table_md, encoding="utf-8")
print(table_md)
print(f"Saved {table_path}")

| Covariate | Electricity | Drones | Flights | Consumer rights | AI & offshoring | Food labels | Vaccines |
|:--------------------------------------------------------------------------------------------|:------------------------------------------------------:|:------------------------------------------:|:-----------------------------------------:|:----------------------------------------------------------------------:|:----------------------------------------------------------------------:|:---------------------------------------------------:|:------------------------------------------------:|
| Sex | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ |
| Age | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ |
| Education | ✓ |  | ✓ | ✓ | ✓ | ✓ | ✓ |
| Income | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ |
| Race | ✓ | ✓ | ✓ |  |  | ✓ | ✓ |
| Hispanic or Latino | ✓ |  |  |  |  | ✓ |  |
| Region | ✓ | ✓ | ✓ |  | ✓ |  | ✓ |
| Place of living |  | ✓ | ✓ |  |  |  |  |
| Religion |  |  | ✓ |  |  |  | ✓ |
| Political party | ✓ |  | ✓ |  | ✓ |  | ✓ |
| Pol